# Train the MRD land-cover model

Explore the training data and model. This calls the same functions as `mrd train`, which publishes the model;
this notebook never uploads anything.

In [ ]:
import pandas as pd

from mrd import CLASSES
from mrd.data import FEATURES
from mrd.train import TRAINING_DATA, cross_validate, drop_unobserved, fit, load_table, read_points

## Training points

In [ ]:
points, null_geometry = read_points(TRAINING_DATA)
print(f"{len(points)} points ({null_geometry} dropped for null geometry)")
points["lulc_code"].value_counts().sort_index()

In [ ]:
labels = {code: f"{code}: {name}" for code, (name, _) in CLASSES.items()}
points.assign(lulc_class=points["lulc_code"].map(labels)).explore(
    column="lulc_class",
    categorical=True,
    categories=list(labels.values()),
    cmap=[colour for _, colour in CLASSES.values()],
    legend=True,
)

## Features at the points

Loaded tile by tile with `mrd.data.load_data`, and cached under `data/cache`.

In [ ]:
raw, not_sampled, cached = load_table(points, 2022, TRAINING_DATA, use_cache=True)
table = drop_unobserved(raw)
print(f"{len(table)} points used; {not_sampled} not sampled, {len(raw) - len(table)} without S2 (cached: {cached})")

## Cross-validation

In [ ]:
cv = cross_validate(table)
for name, report in cv.items():
    print(f"{name}: accuracy {report['overall_accuracy']:.3f}, macro F1 {report['macro_f1']:.3f}")

In [ ]:
matrix = cv["spatial"]["confusion_matrix"]
pd.DataFrame(matrix["matrix"], index=matrix["labels"], columns=matrix["labels"])

## Fit on all points

In [ ]:
model = fit(table)
print(f"OOB score {model.oob_score_:.3f}")
pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)